# Notebook 2: `node1` Cluster Bootstrap

Run this notebook **from inside `node1`** after Terraform provisioning is complete and this repository has been cloned locally on `node1`.

Execution order in this notebook:
1. Prerequisites setup
2. Ansible connectivity check
3. Pre-K8s node preparation
4. Kubespray `cluster.yml` cluster creation
5. Fix GPU node routing
6. Post-K8s configuration
7. GPU node setup and secrets
8. Argo CD application bootstrap

## Prerequisites Setup

Run the setup script first to install required tools (`ansible-core==2.16.4` and `kubectl`).
This must be run before the prerequisites check cell.

In [ ]:
set -euo pipefail
cd "$HOME/recipe-scraper-mlops"
bash devops/scripts/node1_setup.sh

In [ ]:
set -euo pipefail

REPO_ROOT="${REPO_ROOT:-$HOME/recipe-scraper-mlops}"
ANSIBLE_DIR="${ANSIBLE_DIR:-$REPO_ROOT/devops/ansible}"
KUBESPRAY_DIR="${KUBESPRAY_DIR:-$ANSIBLE_DIR/k8s/kubespray}"
KUBESPRAY_INVENTORY="${KUBESPRAY_INVENTORY:-$ANSIBLE_DIR/k8s/inventory/mycluster/hosts.yaml}"

cat <<EOF
Parameter summary:
  REPO_ROOT           = $REPO_ROOT
  ANSIBLE_DIR         = $ANSIBLE_DIR
  KUBESPRAY_DIR       = $KUBESPRAY_DIR
  KUBESPRAY_INVENTORY = $KUBESPRAY_INVENTORY
EOF

In [ ]:
set -euo pipefail

HOST_SHORT="$(hostname -s)"
if [[ "$HOST_SHORT" != node1* ]]; then
  echo "This notebook must run on node1. Current host: $HOST_SHORT" >&2
  exit 1
fi

for cmd in ansible-playbook kubectl ssh bash; do
  command -v "$cmd" >/dev/null || {
    echo "Missing required command: $cmd" >&2
    exit 1
  }
done

[[ -d "$REPO_ROOT" ]] || {
  echo "Repository path not found: $REPO_ROOT" >&2
  exit 1
}

for path in \
  "$ANSIBLE_DIR/inventory.yml" \
  "$ANSIBLE_DIR/general/hello_host.yml" \
  "$ANSIBLE_DIR/pre_k8s/pre_k8s_configure.yml" \
  "$ANSIBLE_DIR/post_k8s/post_k8s_configure.yml" \
  "$ANSIBLE_DIR/post_k8s/fix_gpu_routing.yml" \
  "$ANSIBLE_DIR/argocd/argocd_bootstrap_apps.yml" \
  "$KUBESPRAY_INVENTORY"; do
  [[ -f "$path" ]] || {
    echo "Missing expected file: $path" >&2
    exit 1
  }
done

if [[ ! -f "$KUBESPRAY_DIR/cluster.yml" ]]; then
  echo "Missing Kubespray playbook: $KUBESPRAY_DIR/cluster.yml" >&2
  echo "Initialize submodule if needed: git submodule update --init --recursive" >&2
  exit 1
fi

echo "Prerequisites passed on node1."

## Inventory Drift Checkpoint

Before running playbooks, confirm host IPs in `devops/ansible/inventory.yml` and `devops/ansible/k8s/inventory/mycluster/hosts.yaml` still match Terraform outputs.

## Phase 1: Ansible Connectivity

This notebook uses `id_rsa` as the SSH key for connecting to all cluster nodes. The key must:
1. Be named `id_rsa` in `~/.ssh/` on node1
2. Have no passphrase
3. Be the same key pair registered in Chameleon under your project

Before running the connectivity check, copy your key to node1 from your local machine:

```bash
scp ~/.ssh/id_rsa cc@<node1-floating-ip>:~/.ssh/id_rsa
scp ~/.ssh/id_rsa.pub cc@<node1-floating-ip>:~/.ssh/id_rsa.pub
```

If your key is named differently, update `CHAMELEON_KEY_PATH` in the next cell accordingly.

In [ ]:
set -euo pipefail

# Use id_rsa directly — no separate chameleon key needed
export CHAMELEON_KEY_PATH="$HOME/.ssh/id_rsa"
export ANSIBLE_PRIVATE_KEY_FILE="$CHAMELEON_KEY_PATH"

[[ -f "$CHAMELEON_KEY_PATH" ]] || {
  echo "Missing private key on node1: $CHAMELEON_KEY_PATH" >&2
  exit 1
}

chmod 600 "$CHAMELEON_KEY_PATH"
echo "Using CHAMELEON_KEY_PATH=$CHAMELEON_KEY_PATH"

In [ ]:
set -euo pipefail

cd "$REPO_ROOT"
ANSIBLE_HOST_KEY_CHECKING=False ansible-playbook \
  -i devops/ansible/inventory.yml \
  devops/ansible/general/hello_host.yml \
  -e "ansible_ssh_private_key_file=$HOME/.ssh/id_rsa" \
  --private-key=$HOME/.ssh/id_rsa

## Phase 2: Pre-K8s Preparation

In [ ]:
set -euo pipefail

cd "$REPO_ROOT"
ansible-playbook -i devops/ansible/inventory.yml devops/ansible/pre_k8s/pre_k8s_configure.yml

In [ ]:
set -euo pipefail

cd "$REPO_ROOT"
ansible all -i devops/ansible/inventory.yml -m shell -a 'ip -brief address && ip route' -o

## Phase 3: Kubespray Cluster Creation

Before running `cluster.yml`, pull the Kubespray git submodule from the repo root:

```bash
git submodule update --init --recursive
```

This ensures `devops/ansible/k8s/kubespray/cluster.yml` exists locally.

In [ ]:
set -euo pipefail

# Must run from KUBESPRAY_DIR for role resolution to work
cd "$KUBESPRAY_DIR"
ansible-playbook -i "$KUBESPRAY_INVENTORY" cluster.yml -u cc -b

In [ ]:
set -euo pipefail

if [[ -f "$HOME/.kube/config" ]]; then
  kubectl get nodes -o wide
  kubectl get pods -A
else
  sudo kubectl --kubeconfig /etc/kubernetes/admin.conf get nodes -o wide
  sudo kubectl --kubeconfig /etc/kubernetes/admin.conf get pods -A
fi

## Phase 3.5: Fix GPU Node Routing

Required when `gpu-node-chi` (bare metal CHI@TACC node) is part of the cluster.

Calico assigns a pod CIDR to `gpu-node-chi` but uses VXLAN tunneling that does not work over WireGuard.
This playbook detects the pod CIDR dynamically and adds the correct routes on all nodes so that
cross-node pod communication with `gpu-node-chi` works correctly through WireGuard.

Skip this phase if `gpu-node-chi` is not part of the cluster.

In [ ]:
set -euo pipefail

cd "$REPO_ROOT"
ansible-playbook -i devops/ansible/inventory.yml devops/ansible/post_k8s/fix_gpu_routing.yml

## Phase 4: Post-K8s Configuration

Note: this playbook prints the initial Argo CD admin password in output.

**NVIDIA container toolkit note:** `post_k8s_configure.yml` automatically installs the toolkit on `gpu-node-chi`.
If it hangs (>5 minutes on the toolkit task), cancel it and install manually on the bare metal node:

```bash
ssh -i ~/.ssh/id_rsa cc@<gpu-node-chi-floating-ip>
curl -fsSL https://nvidia.github.io/libnvidia-container/gpgkey | \
  sudo gpg --dearmor -o /usr/share/keyrings/nvidia-container-toolkit-keyring.gpg
curl -s -L https://nvidia.github.io/libnvidia-container/stable/deb/nvidia-container-toolkit.list | \
  sed 's#deb https://#deb [signed-by=/usr/share/keyrings/nvidia-container-toolkit-keyring.gpg] https://#g' | \
  sudo tee /etc/apt/sources.list.d/nvidia-container-toolkit.list
sudo apt-get update
sudo apt-get install -y nvidia-container-toolkit
sudo nvidia-ctk runtime configure --runtime=containerd
sudo systemctl restart containerd
```

Then re-run `post_k8s_configure.yml`.

In [ ]:
set -euo pipefail

cd "$REPO_ROOT"
ansible-playbook -i devops/ansible/inventory.yml devops/ansible/post_k8s/post_k8s_configure.yml

In [ ]:
set -euo pipefail

kubectl get nodes -o wide
argocd version --client
argo version
kubectl get all -n argocd

## Phase 4.5: GPU Node Setup and Secrets

Before running this cell, copy your CHI@TACC openrc file to node1:

```bash
scp ~/path/to/app-cred-openrc.sh cc@<node1-floating-ip>:~/openrc
```

This openrc must have access to the Swift object store where model weights are stored.

Skip this phase if `gpu-node-chi` is not part of the cluster.

In [ ]:
set -euo pipefail

# Create Triton OpenRC secret for model download from Swift
[[ -f "$HOME/openrc" ]] || {
  echo "Missing openrc file at $HOME/openrc" >&2
  echo "Copy it from your local machine: scp ~/path/to/openrc cc@<node1-floating-ip>:~/openrc" >&2
  exit 1
}

kubectl create secret generic triton-openrc \
  --from-file=openrc=$HOME/openrc \
  -n recipe-scraper-platform \
  --dry-run=client -o yaml | kubectl apply -f -

# Label GPU bare metal node
kubectl label node gpu-node-chi node-role=gpu-triton --overwrite
kubectl label node gpu-node-chi nvidia.com/gpu=true --overwrite

# Install Argo Rollouts (required for Triton canary rollout)
kubectl create namespace argo-rollouts --dry-run=client -o yaml | kubectl apply -f -
kubectl apply -n argo-rollouts \
  -f https://github.com/argoproj/argo-rollouts/releases/latest/download/install.yaml

# Wait for Argo Rollouts to be ready
kubectl rollout status deployment/argo-rollouts -n argo-rollouts --timeout=120s

echo "GPU node setup complete."

## Phase 5: Argo CD Bootstrap Apps

In [ ]:
set -euo pipefail

cd "$REPO_ROOT"
ansible-playbook devops/ansible/argocd/argocd_bootstrap_apps.yml

In [ ]:
set -euo pipefail

kubectl get applications -n argocd
kubectl get pods -n argocd
echo "Optional if argocd CLI login already configured: argocd app list && argocd app get platform"

## Recovery Notes (Reruns)

- If a phase fails, fix root cause and rerun only that phase cell (and its validation cell).
- If Kubespray fails mid-run, rerun the same `cluster.yml` command after confirming inventory/SSH reachability.
- If Argo CD apps partially apply, rerun `argocd_bootstrap_apps.yml`; Kubernetes apply is idempotent for unchanged manifests.
- If the NVIDIA toolkit task hangs in post_k8s, install it manually on `gpu-node-chi` (see Phase 4 note) then re-run.
- If Prometheus cannot scrape Triton after cluster restart, re-run `fix_gpu_routing.yml` — Calico resets VXLAN routes on reboot.

## Final Verification + Next Checks

Confirm:
- Cluster nodes are `Ready`.
- Argo CD Applications are registered.
- Core workloads converge to healthy state.
- Triton pod is running on `gpu-node-chi`.

Useful commands:
```bash
kubectl get nodes -o wide
kubectl get pods -A
kubectl get applications -n argocd -w
kubectl get pods -n recipe-scraper-platform
kubectl get rollouts -n recipe-scraper-platform
```

Security note:
- If this notebook output will be shared, clear outputs first because post-K8s phase may include sensitive bootstrap credentials.